In [15]:
from pandas import read_csv
from sklearn.model_selection import train_test_split

![Tafel](../img/tafel_real.png)

In [16]:
data = read_csv("../data/titanic.csv")
## Wären auch möglichkeit den Datensatz zu landen, da er öffentlich zugänglich ist:
# 1) read_csv("titanic.csv")
# 2) read_csv("https://www.tinyurl.com/mci-titanic")

"""
Daten verikal splitten (Input/Output)
"""

# groß X Input und klein y Output
X = data[[
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Embarked'
]]

y = data[[
 'Survived'
]]

In [17]:
"""
Zusammengelegt von mehreren Zellen, da es sich um eine logische Einheit handelt.
"""
X.info()
print(X.shape, y.shape)
X.isna().any()
X.isna().any().any()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    str    
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Embarked  889 non-null    str    
dtypes: float64(1), int64(3), str(2)
memory usage: 41.9 KB
(891, 6) (891, 1)


np.True_

 X["Age"].fillna(999) sollte man vermeiden 

# Daten horizontal splitten (Train/Test) - siehe nächste Zelle

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [19]:
print("Trainings-Daten", X_train.shape, y_train.shape)
print("TestDaten", X_test.shape, y_test.shape)

Trainings-Daten (712, 6) (712, 1)
TestDaten (179, 6) (179, 1)


# Daten Vorverarbeiten
- Skalieren 
- Imputing 
- Encoding; z. B.female und male müssen wir in einen Numerischen Wert verwandeln 

In [20]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [21]:
demo_data = [
    ["male"],
    ["female"],
    ["diverse"],
    ["male"],
    ["female"]
]

In [22]:
encoder = OneHotEncoder()
encoder.fit(demo_data)
encoder.transform(demo_data).toarray()

array([[0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [23]:
demo_data = [
    [1],
    [5],
    [10],
    [15], 
    [1],
    [5],
]

In [24]:
scaler = StandardScaler()
scaler.fit(demo_data)
scaler.transform(demo_data)

array([[-1.03737545],
       [-0.23424607],
       [ 0.76966565],
       [ 1.77357738],
       [-1.03737545],
       [-0.23424607]])

In [27]:
from pandas import DataFrame
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# 1) Spalten automatisch nach Datentyp trennen: numerisch und nicht numerisch
numerische_spalten = X_train.select_dtypes(include=["number"]).columns
kategoriale_spalten = X_train.select_dtypes(exclude=["number"]).columns

# 2) Numerische Daten: fehlende Werte mit Median auffuellen, danach auf 0 bis 1 skalieren
numerische_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

# 3) Kategoriale Daten: fehlende Werte mit "missing" auffuellen, danach One-Hot-Encoding
kategoriale_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("one_hot_encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4) Beide Verarbeitungspfade wieder zu einem Datensatz zusammenfuehren
vorverarbeitung_pipeline = ColumnTransformer(transformers=[
    ("numerisch", numerische_pipeline, numerische_spalten),
    ("kategorial", kategoriale_pipeline, kategoriale_spalten)
])

X_train_verarbeitet = vorverarbeitung_pipeline.fit_transform(X_train)
X_test_verarbeitet = vorverarbeitung_pipeline.transform(X_test)

# Ergebnis wieder in DataFrames umwandeln, damit die Spaltennamen sichtbar bleiben
spaltennamen = vorverarbeitung_pipeline.get_feature_names_out()

X_train_verarbeitet = DataFrame(
    X_train_verarbeitet,
    columns=spaltennamen,
    index=X_train.index
)

X_test_verarbeitet = DataFrame(
    X_test_verarbeitet,
    columns=spaltennamen,
    index=X_test.index
)

print("Numerische Spalten:", list(numerische_spalten))
print("Kategoriale Spalten:", list(kategoriale_spalten))
print("Trainingsdaten nach Vorverarbeitung:", X_train_verarbeitet.shape)
print("Testdaten nach Vorverarbeitung:", X_test_verarbeitet.shape)

X_train_verarbeitet.head()

Numerische Spalten: ['Pclass', 'Age', 'SibSp', 'Parch']
Kategoriale Spalten: ['Sex', 'Embarked']
Trainingsdaten nach Vorverarbeitung: (712, 10)
Testdaten nach Vorverarbeitung: (179, 10)


,numerisch__Pclass,numerisch__Age,numerisch__SibSp,numerisch__Parch,kategorial__Sex_female,kategorial__Sex_male,kategorial__Embarked_C,kategorial__Embarked_Q,kategorial__Embarked_S,kategorial__Embarked_missing
11,0.0,0.723549,0.00,0.000000,1.0,0.0,0.0,0.0,1.0,0.0
266,1.0,0.195778,0.50,0.166667,0.0,1.0,0.0,0.0,1.0,0.0
198,1.0,0.346569,0.00,0.000000,1.0,0.0,0.0,1.0,0.0,0.0
563,1.0,0.346569,0.00,0.000000,0.0,1.0,0.0,0.0,1.0,0.0
565,1.0,0.296306,0.25,0.000000,0.0,1.0,0.0,0.0,1.0,0.0


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numerical_features = make_column_selector(dtype_include='number')
categorical_features = make_column_selector(dtype_exclude='number')

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Trainings-Daten vorverarbeitet:", X_train_prepared.shape)
print("Test-Daten vorverarbeitet:", X_test_prepared.shape)

Trainings-Daten vorverarbeitet: (712, 10)
Test-Daten vorverarbeitet: (179, 10)


In [29]:
preprocessor.fit(X_train)
preprocessor.transform(X_train)

array([[0.        , 0.72354863, 0.        , ..., 0.        , 1.        ,
        0.        ],
       [1.        , 0.19577783, 0.5       , ..., 0.        , 1.        ,
        0.        ],
       [1.        , 0.34656949, 0.        , ..., 1.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.6481528 , 0.125     , ..., 0.        , 0.        ,
        0.        ],
       [0.5       , 0.15807992, 0.        , ..., 0.        , 1.        ,
        0.        ],
       [1.        , 0.34656949, 0.        , ..., 1.        , 0.        ,
        0.        ]], shape=(712, 10))